<div align="center">

# SENDA — Sistema de Gestión de Horas Beca

### Documentación Técnica del Sistema

---

**Universidad Adventista de Centroamérica (UNADECA)**

**Proyecto de Desarrollo de Software**

**Versión:** 1.0 &nbsp;|&nbsp; **Fecha:** Abril 2026

---

</div>

## Tabla de Contenidos

1. [Introducción](#1-introducción)
2. [Descripción General del Sistema](#2-descripción-general-del-sistema)
3. [Stack Tecnológico](#3-stack-tecnológico)
4. [Arquitectura del Sistema](#4-arquitectura-del-sistema)
5. [Modelo de Datos](#5-modelo-de-datos)
6. [Sistema de Autenticación y Seguridad](#6-sistema-de-autenticación-y-seguridad)
7. [API REST — Endpoints del Backend](#7-api-rest)
8. [Módulo: Inicio de Sesión](#8-módulo-inicio-de-sesión)
9. [Módulo: Portal del Estudiante](#9-módulo-portal-del-estudiante)
10. [Módulo: Portal del Jefe de Departamento](#10-módulo-portal-del-jefe-de-departamento)
11. [Módulo: Portal del Administrador](#11-módulo-portal-del-administrador)
12. [Módulo: Portal del Super Administrador](#12-módulo-portal-del-super-administrador)
13. [Módulo: Portal de Contabilidad](#13-módulo-portal-de-contabilidad)
14. [Módulo: Kiosco de Marcaje](#14-módulo-kiosco-de-marcaje)
15. [Sistema de Tiempo Real](#15-sistema-de-tiempo-real)
16. [Lógica de Negocio — Ciclos y Nómina](#16-lógica-de-negocio)
17. [Conclusiones](#17-conclusiones)

---

## 1. Introducción

### 1.1 Contexto y Problemática

La Universidad Adventista de Centroamérica (UNADECA) otorga becas de trabajo a estudiantes que prestan servicio en diferentes departamentos institucionales. Bajo este esquema, los estudiantes realizan labores asignadas y registran las horas trabajadas para recibir un beneficio económico que se aplica a su cuenta estudiantil.

Previo a la implementación de este sistema, el proceso de gestión de horas beca se realizaba de manera **manual**, lo que generaba las siguientes problemáticas:

- **Registros en papel o planillas de Excel** propensos a errores, pérdida de datos y duplicaciones.
- **Falta de trazabilidad**: no existía un historial auditable de quién aprobó o rechazó un registro de horas.
- **Demoras en el procesamiento contable**: la recopilación de datos para la nómina de becas requería consolidación manual de múltiples fuentes.
- **Ausencia de control en tiempo real**: los jefes de departamento no tenían visibilidad inmediata sobre las horas registradas por sus estudiantes.
- **Proceso de marcaje rudimentario**: no existía un mecanismo digital para que los estudiantes registraran entrada y salida directamente desde el departamento.

### 1.2 Objetivo del Sistema

**SENDA** (*Sistema de Gestión de Horas Beca*) fue desarrollado como una plataforma web integral que digitaliza y automatiza todo el ciclo de vida del registro de horas beca, desde el momento en que el estudiante registra su tiempo de trabajo hasta que el departamento de contabilidad procesa el pago correspondiente.

### 1.3 Objetivos Específicos

1. Proveer un sistema de **registro digital de horas** con cronómetro integrado y marcaje por kiosco.
2. Implementar un **flujo de aprobación** con auditoría completa (quién aprobó/rechazó, cuándo y por qué).
3. Ofrecer **reportes contables automatizados** con cálculo de bruto, diezmo y neto por estudiante y departamento.
4. Garantizar la **seguridad de los datos** mediante autenticación JWT y políticas de Row Level Security (RLS) a nivel de base de datos.
5. Proporcionar **actualizaciones en tiempo real** para que los cambios se reflejen instantáneamente en todas las pantallas conectadas.
6. Facilitar la administración de usuarios, departamentos y tarifas desde una interfaz centralizada basada en roles.

---

## 2. Descripción General del Sistema

### 2.1 ¿Qué es SENDA?

SENDA es una **aplicación web de una sola página** (SPA — *Single Page Application*) que permite gestionar el ciclo completo de las horas beca universitarias. El sistema opera bajo un modelo de **cinco roles de usuario**, cada uno con su propio portal y conjunto de permisos:

| Rol | Descripción | Acceso Principal |
|:----|:------------|:-----------------|
| **Estudiante** (`STUDENT`) | Becario que registra y consulta sus horas de trabajo | Cronómetro, historial de horas, resumen financiero |
| **Jefe de Departamento** (`DEPT_HEAD`) | Supervisor que aprueba o rechaza las horas de su equipo | Panel de aprobación, registro de horas, activación de kiosco |
| **Administrador** (`ADMIN`) | Gestiona usuarios, departamentos y la tarifa por hora | Dashboard de KPIs, CRUD de usuarios y departamentos |
| **Super Administrador** (`SUPER_ADMIN`) | Control total del sistema, crea cuentas administrativas | Creación de cuentas ADMIN/CONTABILIDAD, reseteo de contraseñas |
| **Contabilidad** (`ACCOUNTING`) | Procesa la nómina y gestiona las cuentas contables | Reporte de nómina, procesamiento de pagos, configuración contable |

### 2.2 Flujo General del Sistema

El flujo de trabajo de SENDA sigue un proceso lineal con puntos de control:

```
┌─────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  ESTUDIANTE │     │   JEFE DE    │     │              │     │              │
│  Registra   │────▶│ DEPARTAMENTO │────▶│    ADMIN     │────▶│ CONTABILIDAD │
│  sus horas  │     │   Aprueba /  │     │  Supervisa   │     │   Procesa    │
│  (PENDING)  │     │   Rechaza    │     │              │     │  (PROCESSED) │
└─────────────┘     └──────────────┘     └──────────────┘     └──────────────┘
      │                    │                                         │
      │              ┌─────┴─────┐                                   │
      │              │ APPROVED  │                                   │
      │              │     o     │                                   │
      │              │ REJECTED  │                                   │
      │              └───────────┘                                   │
      │                                                              │
      ▼                                                              ▼
  Cronómetro                                                   Nómina con
  o Kiosco                                                   cálculo de
  de marcaje                                                bruto/diezmo/neto
```

### 2.3 Modos de Registro de Horas

SENDA ofrece **dos métodos** para que los estudiantes registren su tiempo de trabajo:

1. **Registro Manual (Cronómetro):** El estudiante inicia un cronómetro desde su portal personal, trabaja, y al finalizar detiene el cronómetro. El sistema calcula automáticamente las horas transcurridas y genera un registro con estado `PENDING`.

2. **Kiosco de Marcaje:** En cada departamento se puede activar una estación de marcaje (kiosco) en una computadora compartida. Los estudiantes se identifican con su carnet y contraseña para registrar entrada (*clock-in*) y salida (*clock-out*). El sistema calcula las horas trabajadas automáticamente.

---

## 3. Stack Tecnológico

El sistema SENDA fue construido con una arquitectura moderna de **separación completa entre frontend y backend** (*decoupled architecture*), seleccionando tecnologías reconocidas en la industria por su rendimiento, seguridad y escalabilidad.

### 3.1 Frontend — Interfaz de Usuario

| Tecnología | Versión | Propósito |
|:-----------|:--------|:----------|
| **React** | 19.2 | Biblioteca principal para la construcción de interfaces de usuario basadas en componentes |
| **TypeScript** | 5.8 | Superconjunto tipado de JavaScript que provee verificación estática de tipos en tiempo de compilación |
| **Vite** | 6.4 | Herramienta de construcción (*build tool*) de nueva generación con recarga instantánea en desarrollo (*HMR*) |
| **Tailwind CSS** | 4.2 | Framework de CSS utilitario que permite diseñar directamente en el markup sin salir del componente |
| **Recharts** | 3.3 | Biblioteca de gráficos basada en React y D3.js para la visualización de datos estadísticos |
| **Motion** (Framer Motion) | 12.x | Motor de animaciones declarativas para transiciones fluidas entre pantallas y componentes |
| **Sonner** | 2.x | Sistema de notificaciones *toast* no bloqueantes para feedback al usuario |
| **jsPDF + AutoTable** | 4.2 / 5.0 | Generación de documentos PDF directamente en el navegador para reportes contables |
| **Lucide React** | 0.575 | Conjunto de iconos SVG optimizados como componentes React |
| **Supabase JS** | 2.98 | Cliente oficial para interactuar con la API de Supabase (Realtime, Auth) desde el frontend |

**¿Por qué React + TypeScript?**
React fue elegido por ser la biblioteca de UI más adoptada en la industria, con un ecosistema maduro de herramientas y una comunidad activa. TypeScript añade una capa de seguridad al código al detectar errores de tipos antes de la ejecución, lo que reduce significativamente los bugs en producción. Juntos, permiten construir interfaces complejas con código mantenible y autocompletado inteligente en el editor.

**¿Por qué Vite?**
A diferencia de herramientas tradicionales como Webpack, Vite aprovecha los módulos nativos de ES del navegador para ofrecer un servidor de desarrollo que inicia en milisegundos y recarga solo los módulos modificados (*Hot Module Replacement*), acelerando drásticamente el ciclo de desarrollo.

### 3.2 Backend — Servidor de Aplicación

| Tecnología | Versión | Propósito |
|:-----------|:--------|:----------|
| **Node.js** | 24.x | Entorno de ejecución de JavaScript del lado del servidor, basado en el motor V8 de Chrome |
| **Express** | 4.21 | Framework minimalista de Node.js para construir APIs REST con middleware composable |
| **ES Modules (ESM)** | Nativo | Sistema de módulos estándar de JavaScript (`import`/`export`) en lugar del antiguo CommonJS |

**¿Por qué Express?**
Express es el framework web más utilizado en el ecosistema Node.js. Su arquitectura basada en middleware permite agregar funcionalidades (autenticación, CORS, logging) de forma modular y desacoplada. Para un sistema que requiere una API REST clara y bien definida, Express ofrece la flexibilidad necesaria sin la complejidad de frameworks más pesados.

**¿Por qué ESM puro (archivos `.mjs`)?**
El backend utiliza módulos ES nativos (archivos `.mjs`) en lugar de CommonJS (`.cjs`). Esto alinea el código del servidor con los estándares modernos de JavaScript, facilita el *tree-shaking* (eliminación de código muerto) y permite compartir patrones de importación con el frontend.

### 3.3 Base de Datos y Servicios Backend

| Tecnología | Propósito |
|:-----------|:----------|
| **Supabase** | Plataforma Backend-as-a-Service construida sobre PostgreSQL |
| **PostgreSQL** | Motor de base de datos relacional de código abierto con soporte avanzado de tipos, funciones y triggers |
| **Row Level Security (RLS)** | Mecanismo nativo de PostgreSQL que restringe el acceso a filas según el contexto del usuario autenticado |
| **Supabase Auth** | Sistema de autenticación basado en JWT que integra usuarios con las políticas RLS de la base de datos |
| **Supabase Realtime** | Servicio de suscripción a cambios en la base de datos vía WebSockets para actualizaciones en vivo |

**¿Por qué Supabase?**
Supabase fue seleccionado por combinar en una sola plataforma: base de datos PostgreSQL gestionada, autenticación JWT integrada, APIs auto-generadas, y un sistema de suscripciones en tiempo real. Esto elimina la necesidad de configurar y mantener estos servicios por separado, acelerando significativamente el desarrollo.

**¿Por qué RLS en lugar de validación solo en el backend?**
La seguridad a nivel de fila (RLS) proporciona una **capa de defensa adicional directamente en la base de datos**. Incluso si un atacante lograra evadir la validación del backend, las políticas RLS de PostgreSQL impedirían el acceso no autorizado a los datos. Este modelo de seguridad en profundidad (*defense in depth*) es considerado una buena práctica en sistemas que manejan datos sensibles.

### 3.4 Herramientas de Desarrollo y Calidad

| Herramienta | Propósito |
|:------------|:----------|
| **ESLint** 9.x | Análisis estático de código para detectar patrones problemáticos y forzar convenciones |
| **Vitest** 4.x | Framework de testing unitario compatible con Vite, con recarga rápida y cobertura integrada |
| **Husky** + **lint-staged** | Hooks de Git que ejecutan linting automático antes de cada commit para mantener la calidad del código |
| **TypeScript Compiler (`tsc`)** | Verificación de tipos sin emitir código (`--noEmit`) como paso de CI/CD |

### 3.5 Puertos y Comunicación

```
┌──────────────────┐          ┌──────────────────┐          ┌──────────────────┐
│                  │  HTTP    │                  │  SQL     │                  │
│     FRONTEND     │─────────▶│     BACKEND      │─────────▶│    SUPABASE      │
│   (Vite :3000)   │  /api/v1 │  (Express :4000) │  JWT     │  (PostgreSQL)    │
│                  │◀─────────│                  │◀─────────│                  │
│                  │  JSON    │                  │  Rows    │                  │
└──────────────────┘          └──────────────────┘          └──────────────────┘
         │                                                           │
         │                    WebSocket (wss://)                     │
         └──────────────────────────────────────────────────────────┘
                         Supabase Realtime
```

- **Puerto 3000** (Frontend): Servidor de desarrollo Vite con proxy inverso hacia el backend.
- **Puerto 4000** (Backend): API REST Express que procesa la lógica de negocio y se comunica con Supabase.
- **Supabase Cloud**: Base de datos PostgreSQL gestionada con Realtime habilitado para las tablas `profiles`, `departments`, `work_logs` y `hourly_rates`.

---

## 4. Arquitectura del Sistema

### 4.1 Patrón Arquitectónico

SENDA implementa una **arquitectura en capas** (*layered architecture*) combinada con un **diseño basado en características** (*feature-based architecture*) en el backend. Este enfoque organiza el código por dominio de negocio en lugar de por tipo de archivo, lo que facilita la mantenibilidad y escalabilidad del sistema.

### 4.2 Arquitectura del Backend

Cada módulo funcional del backend sigue una estructura de **cinco capas** con responsabilidades claramente definidas:

```
backend/src/features/<modulo>/
  ├── <modulo>.schemas.mjs      ← Constantes, validaciones, regexes del dominio
  ├── <modulo>.repository.mjs   ← Consultas a Supabase (solo I/O de datos)
  ├── <modulo>.service.mjs      ← Lógica de negocio + validación de permisos
  ├── <modulo>.controller.mjs   ← Extrae parámetros del request, invoca al service
  └── <modulo>.routes.mjs       ← Definición de rutas Express con middleware
```

**Responsabilidades por capa:**

| Capa | Responsabilidad | Ejemplo |
|:-----|:----------------|:--------|
| **Schemas** | Define constantes del dominio, expresiones regulares de validación y funciones auxiliares puras | `WORK_LOG_STATUSES`, `VALID_ROLES`, regex de email institucional |
| **Repository** | Encapsula todas las consultas a la base de datos. Solo realiza operaciones de entrada/salida (I/O) | `findAll()`, `insert()`, `updateStatus()` |
| **Service** | Contiene la lógica de negocio: validación de datos, verificación de permisos y orquestación de operaciones | Verificar que un `DEPT_HEAD` solo apruebe horas de su departamento |
| **Controller** | Extrae los datos del objeto `request` de Express, invoca al service correspondiente y retorna la respuesta HTTP | Extraer `req.body`, llamar `service.createWorkLog()`, retornar `res.status(201).json()` |
| **Routes** | Define las rutas del módulo, aplica middleware de autenticación y conecta con los controllers | `router.post('/', requireAuth, controller.create)` |

**Principio de separación:** La capa Repository **nunca** contiene lógica de negocio, y la capa Service **nunca** accede directamente al objeto `req`/`res` de Express. Esta separación permite probar cada capa de forma independiente.

### 4.3 Arquitectura del Frontend

El frontend organiza su código en las siguientes capas:

```
frontend/
  ├── api/                 ← Cliente HTTP centralizado (apiClient.ts)
  ├── services/            ← Funciones que consumen la API REST del backend
  ├── hooks/               ← Custom hooks de React (estado, datos, lógica de UI)
  ├── screens/             ← Pantallas principales organizadas por rol
  │     ├── student/
  │     ├── admin/
  │     ├── depthead/
  │     ├── superadmin/
  │     ├── accounting/
  │     └── kiosk/
  ├── components/          ← Componentes reutilizables (UI, layout)
  ├── lib/                 ← Funciones utilitarias (business logic, formateo)
  └── types.ts             ← Definición de tipos e interfaces TypeScript
```

**Flujo de datos en el frontend:**

```
   Pantalla (Screen)
        │
        ▼
   Hook personalizado (useWorkLogs, useUsers, etc.)
        │
        ▼
   Service (workLogService.ts)
        │
        ▼
   apiClient.ts → fetch HTTP → Backend Express
```

### 4.4 Módulos del Sistema

El backend implementa **ocho módulos funcionales**, cada uno como un directorio independiente:

| Módulo | Directorio | Responsabilidad |
|:-------|:-----------|:----------------|
| **Autenticación** | `features/auth/` | Login, logout, sesión activa, generación de JWT |
| **Usuarios** | `features/users/` | CRUD de perfiles, creación en Supabase Auth, reseteo de contraseñas |
| **Departamentos** | `features/departments/` | CRUD de departamentos, asignación de jefes y centros de costo |
| **Bitácoras de Horas** | `features/workLogs/` | Registro, consulta y cambio de estado de horas trabajadas |
| **Kiosco** | `features/kiosk/` | Activación/desactivación de estaciones de marcaje, clock-in/out |
| **Tarifas** | `features/rates/` | Consulta y actualización de la tarifa por hora vigente |
| **Reportes** | `features/reports/` | Generación de reportes de nómina y detalle por estudiante |
| **Contabilidad** | `features/accounting/` | Configuración de cuentas contables y cuentas por cobrar |

### 4.5 Código Compartido

El directorio `shared/` contiene utilidades transversales al sistema:

| Archivo | Propósito |
|:--------|:----------|
| `middleware/requireAuth.mjs` | Middleware Express que valida el token JWT y adjunta el usuario autenticado al request |
| `utils/mappers.mjs` | Funciones de conversión entre formato `snake_case` (base de datos) y `camelCase` (API) |
| `utils/normalize.mjs` | Normalización de texto, timestamps ISO y construcción de emails internos de autenticación |
| `errors/AppError.mjs` | Clase de error personalizada con código HTTP y código de error semántico |

---

## 5. Modelo de Datos

### 5.1 Diagrama Entidad-Relación

El sistema SENDA utiliza **PostgreSQL** como motor de base de datos, gestionado a través de Supabase. El esquema consta de **8 tablas principales** y **2 tipos enumerados** personalizados, diseñados para garantizar la integridad referencial y la trazabilidad completa de las operaciones.

```
┌──────────────────┐       ┌──────────────────┐
│   auth.users     │       │   departments    │
│   (Supabase)     │       │                  │
│──────────────────│       │──────────────────│
│ id (UUID) PK     │       │ id (UUID) PK     │
│ email            │       │ name (UNIQUE)    │
│ encrypted_pass   │       │ head_id → profiles│
│                  │       │ cost_center      │
└────────┬─────────┘       └────────┬─────────┘
         │ 1:1                      │
         ▼                          │
┌──────────────────┐                │
│    profiles      │                │
│──────────────────│                │
│ id (UUID) PK/FK  │                │
│ name             │                │
│ role (ENUM)      │◀───────────────┘
│ carnet (UNIQUE)  │           1:N
│ employee_number  │
│ department_id ───┼───────────────▶ departments.id
│ is_active        │
│ institutional_   │
│   email (UNIQUE) │
└────────┬─────────┘
         │ 1:N
         ▼
┌──────────────────┐      ┌──────────────────┐
│   work_logs      │      │   kiosk_state    │
│──────────────────│      │──────────────────│
│ id (UUID) PK     │      │ id (UUID) PK     │
│ student_id ──────┼──▶   │ department_id ───┼──▶ departments.id (UNIQUE)
│ department_id ───┼──▶   │ activated_by ────┼──▶ profiles.id
│ date (DATE)      │      │ activated_at     │
│ hours (0-12)     │      │ shifts (JSONB)   │
│ description      │      └────────┬─────────┘
│ status (ENUM)    │               │ 1:N
│ entry_source     │               ▼
│ start_time       │      ┌──────────────────┐
│ end_time         │      │ kiosk_sessions   │
│ approved_by ─────┼──▶   │──────────────────│
│ approved_at      │      │ id (UUID) PK     │
│ rejected_by ─────┼──▶   │ kiosk_id ────────┼──▶ kiosk_state.id
│ rejected_at      │      │ student_id ──────┼──▶ profiles.id
│ rejection_reason │      │ started_at       │
└──────────────────┘      └──────────────────┘

┌──────────────────┐      ┌──────────────────┐      ┌──────────────────────┐
│  hourly_rates    │      │ accounting_config│      │ student_receivables  │
│──────────────────│      │──────────────────│      │──────────────────────│
│ id (UUID) PK     │      │ id (INT, =1) PK  │      │ id (UUID) PK         │
│ rate (> 0)       │      │ becas_account    │      │ student_id ──────────┼──▶
│ effective_date   │      │ becas_name       │      │ period_key           │
│ created_by ──────┼──▶   │ diezmo_account   │      │ amount (>= 0)        │
│                  │      │ diezmo_name      │      │ created_by ──────────┼──▶
└──────────────────┘      │ payable_account  │      └──────────────────────┘
                          │ payable_name     │
                          │ receivable_acct  │
                          │ receivable_name  │
                          └──────────────────┘
```

### 5.2 Tipos Enumerados

El esquema define dos tipos `ENUM` de PostgreSQL para garantizar que solo se almacenen valores válidos:

**`user_role`** — Roles del sistema:
| Valor | Descripción |
|:------|:------------|
| `SUPER_ADMIN` | Administrador con control total del sistema |
| `ADMIN` | Administrador operativo de usuarios y departamentos |
| `DEPT_HEAD` | Jefe de departamento con capacidad de aprobación |
| `STUDENT` | Estudiante becario que registra horas |
| `ACCOUNTING` | Personal de contabilidad que procesa la nómina |

**`work_log_status`** — Estados del ciclo de vida de una bitácora:
| Valor | Descripción |
|:------|:------------|
| `PENDING` | Registrada por el estudiante, en espera de revisión |
| `APPROVED` | Aprobada por el jefe de departamento |
| `REJECTED` | Rechazada con motivo obligatorio |
| `PROCESSED` | Procesada por contabilidad para el pago |

### 5.3 Tablas Principales

#### **`profiles`** — Perfiles de Usuario

Tabla vinculada 1:1 con `auth.users` de Supabase. Al crear un usuario en Supabase Auth, un trigger automático (`handle_new_user`) crea el perfil correspondiente en esta tabla.

| Columna | Tipo | Restricciones | Descripción |
|:--------|:-----|:-------------|:------------|
| `id` | UUID | PK, FK → auth.users | Identificador único heredado de Supabase Auth |
| `name` | TEXT | NOT NULL | Nombre completo del usuario |
| `role` | user_role | NOT NULL, DEFAULT 'STUDENT' | Rol asignado al usuario |
| `carnet` | TEXT | UNIQUE, NULLABLE | Número de carnet (solo estudiantes) |
| `employee_number` | TEXT | UNIQUE, NULLABLE | Número de empleado (solo jefes/admin) |
| `institutional_email` | TEXT | UNIQUE, NULLABLE | Correo electrónico institucional |
| `department_id` | UUID | FK → departments | Departamento asignado |
| `is_active` | BOOLEAN | DEFAULT TRUE | Indica si la cuenta está activa |
| `created_at` | TIMESTAMPTZ | Auto | Fecha de creación |
| `updated_at` | TIMESTAMPTZ | Trigger | Se actualiza automáticamente con cada modificación |

**Restricciones de integridad:**
- Los estudiantes (`STUDENT`) requieren `carnet` y `department_id`.
- Los jefes de departamento (`DEPT_HEAD`) requieren `employee_number`.
- El correo institucional, si se proporciona, debe cumplir un formato válido de email.

#### **`departments`** — Departamentos Institucionales

| Columna | Tipo | Restricciones | Descripción |
|:--------|:-----|:-------------|:------------|
| `id` | UUID | PK | Identificador único |
| `name` | TEXT | NOT NULL, UNIQUE | Nombre del departamento |
| `head_id` | UUID | FK → profiles, ON DELETE SET NULL | Jefe asignado |
| `cost_center` | TEXT | UNIQUE (si presente) | Centro de costo contable (formato `NN-NN-NN`) |
| `created_at` / `updated_at` | TIMESTAMPTZ | Auto / Trigger | Timestamps de auditoría |

#### **`work_logs`** — Bitácoras de Horas Trabajadas

Tabla central del sistema que almacena cada registro de horas con trazabilidad completa de aprobación.

| Columna | Tipo | Restricciones | Descripción |
|:--------|:-----|:-------------|:------------|
| `id` | UUID | PK | Identificador único |
| `student_id` | UUID | FK → profiles, NOT NULL | Estudiante que realizó las horas |
| `department_id` | UUID | FK → departments, NOT NULL | Departamento donde se trabajó |
| `date` | DATE | NOT NULL | Fecha del trabajo |
| `hours` | NUMERIC(5,2) | CHECK: > 0 AND ≤ 12 | Horas trabajadas (máximo 12 por registro) |
| `description` | TEXT | NOT NULL, max 200 chars | Descripción de las tareas realizadas |
| `status` | work_log_status | NOT NULL, DEFAULT 'PENDING' | Estado actual del registro |
| `entry_source` | TEXT | DEFAULT 'MANUAL' | Origen del registro: `MANUAL` o `KIOSK` |
| `start_time` / `end_time` | TIMESTAMPTZ | NULLABLE | Timestamps de inicio/fin (registros por kiosco) |
| `approved_by` | UUID | FK → profiles | Quién aprobó el registro |
| `approved_at` | TIMESTAMPTZ | — | Cuándo fue aprobado |
| `rejected_by` | UUID | FK → profiles | Quién rechazó el registro |
| `rejected_at` | TIMESTAMPTZ | — | Cuándo fue rechazado |
| `rejection_reason` | TEXT | Requerido si status = REJECTED | Motivo del rechazo |

**Restricciones de integridad:**
- `approved_by` y `approved_at` deben ser ambos nulos o ambos no nulos.
- `rejected_by` y `rejected_at` deben ser ambos nulos o ambos no nulos.
- `end_time >= start_time` cuando ambos están presentes.
- `rejection_reason` es obligatorio únicamente cuando `status = 'REJECTED'`.

### 5.4 Migraciones

El esquema de la base de datos evoluciona mediante **migraciones versionadas** almacenadas en `backend/supabase/migrations/`:

| Migración | Descripción |
|:----------|:------------|
| `001_senda_full_schema` | Esquema completo inicial: tablas, enums, funciones, triggers, RLS |
| `002_profiles_is_active` | Agrega columna `is_active` a perfiles para desactivación lógica |
| `003_admin_requirements_and_worklog_audit` | Relaja restricciones de admin, agrega campos de auditoría a work_logs |
| `004_enable_realtime_for_core_tables` | Habilita replicación en tiempo real para tablas críticas |
| `005_accounting_config_and_cost_center` | Agrega tabla de configuración contable y centros de costo |
| `006_student_receivables` | Tabla de cuentas por cobrar por estudiante y período |
| `007_relax_cost_center_constraints` | Flexibiliza formato de centro de costo a `NN-NN-NN` |

---

## 6. Seguridad y Autenticación

### 6.1 Flujo de Autenticación

SENDA implementa un sistema de autenticación basado en **JSON Web Tokens (JWT)** gestionado por Supabase Auth. El flujo completo es el siguiente:

```
┌─────────────┐     POST /api/auth/login      ┌─────────────┐     signInWithPassword()     ┌─────────────┐
│  Frontend   │ ──────────────────────────────▶│   Backend   │ ──────────────────────────────▶│  Supabase   │
│  (React)    │     { identifier, password }   │  (Express)  │     { email, password }       │   Auth      │
│             │◀────────────────────────────── │             │◀────────────────────────────── │             │
│             │     { token, user }            │             │     { access_token, user }     │             │
└──────┬──────┘                                └─────────────┘                                └─────────────┘
       │
       │  Almacena token en
       │  localStorage('senda_token')
       │
       ▼
  ┌──────────────────────────────────────────────────────┐
  │  Todas las peticiones subsiguientes incluyen:        │
  │  Authorization: Bearer <token>                       │
  │                                                      │
  │  El backend valida el token con supabase.auth.getUser()│
  │  y extrae el perfil del usuario autenticado          │
  └──────────────────────────────────────────────────────┘
```

**Proceso paso a paso:**

1. El usuario ingresa su **carnet** (estudiantes), **número de empleado** (jefes) o **login especial** (administradores).
2. El frontend envía las credenciales al endpoint `POST /api/auth/login`.
3. El backend construye el correo electrónico institucional a partir del identificador (ej. `carnet@unadeca.work.senda`).
4. Se autentica contra Supabase Auth con `signInWithPassword()`.
5. Si es exitoso, el backend retorna el token JWT y los datos del perfil.
6. El frontend almacena el token en `localStorage` bajo la clave `senda_token` a través de la clase `TokenManager`.
7. Todas las peticiones posteriores incluyen el header `Authorization: Bearer <token>`.

### 6.2 Middleware de Autenticación

El backend protege las rutas mediante el middleware `requireAuth`, ubicado en `backend/src/shared/middleware/auth.mjs`:

```javascript
// Pseudocódigo del middleware requireAuth
async function requireAuth(req, res, next) {
  // 1. Extrae el token del header Authorization
  const token = req.headers.authorization?.replace('Bearer ', '');

  // 2. Valida el token con Supabase Auth
  const { data: { user }, error } = await supabase.auth.getUser(token);

  // 3. Busca el perfil completo del usuario
  const profile = await getProfileById(user.id);

  // 4. Verifica que la cuenta esté activa
  if (!profile.is_active) throw new ForbiddenError();

  // 5. Adjunta el perfil a req.user para los controllers
  req.user = profile;
  next();
}
```

### 6.3 Row-Level Security (RLS)

Supabase implementa **Row-Level Security** (Seguridad a Nivel de Fila) directamente en PostgreSQL, garantizando que incluso si un atacante obtiene acceso directo a la API de Supabase, solo pueda acceder a los datos permitidos por su rol.

**Políticas principales del sistema:**

| Tabla | Política | Condición de acceso |
|:------|:---------|:-------------------|
| `profiles` | SELECT | El usuario puede ver su propio perfil (`auth.uid() = id`) |
| `profiles` | UPDATE | El usuario solo puede actualizar su propio perfil |
| `departments` | SELECT | Lectura permitida para todos los usuarios autenticados |
| `work_logs` | ALL | Permiso completo si `student_id = auth.uid()` |
| `work_logs` | SELECT (jefe) | El jefe puede ver logs de su departamento |
| `work_logs` | UPDATE (jefe) | El jefe puede aprobar/rechazar logs de su departamento |
| `hourly_rates` | SELECT | Lectura pública para usuarios autenticados |
| `hourly_rates` | INSERT/UPDATE | Solo roles administrativos (`ADMIN`, `SUPER_ADMIN`) |
| `kiosk_state` | ALL | Basado en `department_id` y rol del usuario |

### 6.4 Matriz de Permisos por Rol

La capa de servicio en el backend aplica una **segunda verificación** de permisos, independiente de RLS, para operaciones sensibles:

| Operación | SUPER_ADMIN | ADMIN | DEPT_HEAD | STUDENT | ACCOUNTING |
|:----------|:----------:|:-----:|:---------:|:-------:|:----------:|
| **Ver todos los usuarios** | ✅ | ✅ | ❌ | ❌ | ❌ |
| **Crear usuarios** | ✅ | ✅ | ❌ | ❌ | ❌ |
| **Editar cualquier usuario** | ✅ | ✅* | ❌ | ❌ | ❌ |
| **Desactivar usuarios** | ✅ | ✅* | ❌ | ❌ | ❌ |
| **Gestionar departamentos** | ✅ | ✅ | ❌ | ❌ | ❌ |
| **Ver todos los work logs** | ✅ | ✅ | ❌ | ❌ | ❌ |
| **Crear work logs propios** | ❌ | ❌ | ❌ | ✅ | ❌ |
| **Aprobar/Rechazar logs** | ❌ | ❌ | ✅** | ❌ | ❌ |
| **Modificar tasa por hora** | ✅ | ✅ | ❌ | ❌ | ❌ |
| **Ver reporte de nómina** | ✅ | ❌ | ❌ | ❌ | ✅ |
| **Marcar como procesado** | ❌ | ❌ | ❌ | ❌ | ✅ |
| **Activar modo kiosco** | ❌ | ❌ | ✅ | ❌ | ❌ |
| **Gestionar config contable** | ✅ | ❌ | ❌ | ❌ | ✅ |

\* ADMIN no puede modificar usuarios con rol `SUPER_ADMIN`.
\*\* DEPT_HEAD solo puede aprobar/rechazar logs de **su propio departamento**.

### 6.5 Medidas de Seguridad Adicionales

| Medida | Implementación |
|:-------|:---------------|
| **Hashing de contraseñas** | Supabase Auth usa `bcrypt` internamente |
| **Tokens con expiración** | JWT con tiempo de vida limitado configurado en Supabase |
| **Validación de entrada** | Schemas de validación en la capa de servicio del backend |
| **SERVICE_ROLE_KEY aislada** | `adminSupabase` (con permisos elevados) solo existe en el backend, nunca expuesta al frontend |
| **Desactivación lógica** | Los usuarios desactivados (`is_active = false`) no pueden autenticarse |
| **CORS configurado** | Solo el origen del frontend está permitido para peticiones cross-origin |
| **Auditoría de acciones** | Los campos `approved_by`, `rejected_by`, `created_by` registran quién realizó cada acción |

---

## 7. API REST

### 7.1 Diseño General

La API de SENDA sigue los principios de diseño **RESTful** con las siguientes convenciones:

- **Versionado**: Todas las rutas están bajo el prefijo `/api/v1/`, permitiendo evolución futura sin romper compatibilidad.
- **Autenticación**: 28 de 31 endpoints requieren el middleware `requireAuth` (token JWT en el header `Authorization`).
- **Formato de respuesta**: JSON con estructura consistente.
- **Códigos HTTP**: Se utilizan códigos estándar (200 OK, 201 Created, 400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found, 409 Conflict).
- **Manejo de errores centralizado**: Middleware `errorHandler` captura todas las excepciones y retorna un objeto JSON uniforme.

### 7.2 Mapa Completo de Endpoints

El sistema expone **31 endpoints** organizados en 8 módulos funcionales.

#### Autenticación (`/api/v1/auth`)

| Método | Ruta | Auth | Descripción |
|:------:|:-----|:----:|:------------|
| `POST` | `/api/v1/auth/login` | ❌ | Inicio de sesión, retorna token JWT y datos del perfil |
| `GET` | `/api/v1/auth/me` | ✅ | Obtiene el perfil del usuario autenticado |
| `POST` | `/api/v1/auth/logout` | ❌ | Cierra la sesión (invalida el token) |

#### Usuarios (`/api/v1/users`)

| Método | Ruta | Auth | Descripción |
|:------:|:-----|:----:|:------------|
| `GET` | `/api/v1/users` | ✅ | Lista todos los usuarios (Admin/SuperAdmin) |
| `POST` | `/api/v1/users` | ✅ | Crea un nuevo usuario con su cuenta en Supabase Auth |
| `PATCH` | `/api/v1/users/:id` | ✅ | Actualiza datos de un usuario (nombre, rol, departamento) |
| `DELETE` | `/api/v1/users/:id` | ✅ | Desactiva un usuario (borrado lógico) |
| `POST` | `/api/v1/users/:id/reset-password` | ✅ | Restablece la contraseña del usuario |

#### Departamentos (`/api/v1/departments`)

| Método | Ruta | Auth | Descripción |
|:------:|:-----|:----:|:------------|
| `GET` | `/api/v1/departments` | ✅ | Lista todos los departamentos con su jefe asignado |
| `POST` | `/api/v1/departments` | ✅ | Crea un nuevo departamento |
| `PATCH` | `/api/v1/departments/:id` | ✅ | Actualiza nombre, jefe o centro de costo |
| `DELETE` | `/api/v1/departments/:id` | ✅ | Elimina un departamento |

#### Bitácoras de Trabajo (`/api/v1/work-logs`)

| Método | Ruta | Auth | Descripción |
|:------:|:-----|:----:|:------------|
| `GET` | `/api/v1/work-logs` | ✅ | Lista registros con filtros por estudiante, estado, ciclo |
| `POST` | `/api/v1/work-logs` | ✅ | Crea un nuevo registro de horas (solo estudiantes) |
| `PATCH` | `/api/v1/work-logs/:id/status` | ✅ | Cambia el estado de un registro individual |
| `PATCH` | `/api/v1/work-logs/bulk-status` | ✅ | Actualización masiva de estados (aprobación/procesamiento en lote) |

#### Kiosco (`/api/v1/kiosk`)

| Método | Ruta | Auth | Descripción |
|:------:|:-----|:----:|:------------|
| `GET` | `/api/v1/kiosk/:departmentId` | ✅ | Obtiene el estado del kiosco de un departamento |
| `POST` | `/api/v1/kiosk/activate` | ✅ | Activa el modo kiosco para un departamento |
| `POST` | `/api/v1/kiosk/deactivate` | ✅ | Desactiva el kiosco y finaliza sesiones activas |
| `POST` | `/api/v1/kiosk/clock-in` | ✅ | Registra la entrada de un estudiante |
| `POST` | `/api/v1/kiosk/clock-out` | ✅ | Registra la salida, calcula horas y crea work log |
| `POST` | `/api/v1/kiosk/cancel-session` | ✅ | Cancela una sesión activa (crea log como REJECTED) |
| `PATCH` | `/api/v1/kiosk/shifts` | ✅ | Configura los turnos programados del kiosco |

#### Tasa por Hora (`/api/v1/rate`)

| Método | Ruta | Auth | Descripción |
|:------:|:-----|:----:|:------------|
| `GET` | `/api/v1/rate` | ✅ | Obtiene la tasa vigente y su fecha efectiva |
| `PUT` | `/api/v1/rate` | ✅ | Establece una nueva tasa por hora (Admin/SuperAdmin) |

#### Reportes (`/api/v1/reports`)

| Método | Ruta | Auth | Descripción |
|:------:|:-----|:----:|:------------|
| `GET` | `/api/v1/reports/payroll` | ✅ | Genera reporte de nómina filtrado por ciclo o trimestre |
| `GET` | `/api/v1/reports/student/:studentId` | ✅ | Reporte detallado de un estudiante específico |

#### Contabilidad (`/api/v1/accounting`)

| Método | Ruta | Auth | Descripción |
|:------:|:-----|:----:|:------------|
| `GET` | `/api/v1/accounting/config` | ✅ | Obtiene la configuración de cuentas contables |
| `PUT` | `/api/v1/accounting/config` | ✅ | Actualiza las cuentas contables (becas, diezmo, etc.) |
| `PUT` | `/api/v1/accounting/receivables` | ✅ | Registra o actualiza cuentas por cobrar de estudiantes |

#### Utilidad

| Método | Ruta | Auth | Descripción |
|:------:|:-----|:----:|:------------|
| `GET` | `/health` | ❌ | Verificación de disponibilidad del servidor |

### 7.3 Resumen Estadístico

| Métrica | Valor |
|:--------|:------|
| Total de endpoints | 31 |
| Endpoints protegidos | 28 (90%) |
| Endpoints públicos | 3 (health, login, logout) |
| Métodos GET | 10 |
| Métodos POST | 10 |
| Métodos PATCH | 5 |
| Métodos PUT | 3 |
| Métodos DELETE | 2 |

---

## 8. Módulo de Inicio de Sesión

### 8.1 Descripción General

La pantalla de inicio de sesión es el punto de entrada único para todos los roles del sistema. Implementa un diseño responsivo de dos columnas: branding institucional (izquierda, oculto en dispositivos móviles) y formulario de autenticación (derecha).

### 8.2 Interfaz de Usuario

```
┌────────────────────────────────┬────────────────────────────────┐
│                                │                                │
│   ─── UNADECA ───              │     ┌──────────────────────┐   │
│                                │     │  🔒 SENDA             │   │
│   • Gestión inteligente        │     │  Sistema de Control   │   │
│   • Registro automatizado     │     │  de Horas Becarias    │   │
│                                │     ├──────────────────────┤   │
│  [Branding institucional]      │     │  Usuario / Carnet     │   │
│                                │     │  ┌──────────────────┐ │   │
│                                │     │  │                  │ │   │
│                                │     │  └──────────────────┘ │   │
│                                │     │  Contraseña       👁  │   │
│                                │     │  ┌──────────────────┐ │   │
│                                │     │  │  ••••••••        │ │   │
│                                │     │  └──────────────────┘ │   │
│                                │     │                      │   │
│                                │     │  [Entrar al Sistema →]│   │
│                                │     └──────────────────────┘   │
│                                │                                │
└────────────────────────────────┴────────────────────────────────┘
```

### 8.3 Flujo de Autenticación por Rol

El campo de identificador acepta diferentes formatos según el tipo de usuario:

| Rol | Identificador | Ejemplo |
|:----|:-------------|:--------|
| Estudiante | Número de carnet | `12345` |
| Jefe de Departamento | Número de empleado | `EMP001` |
| Administrador | Login especial | `admin` |
| Super Administrador | Login especial | `superadmin` |
| Contabilidad | Login especial | `contabilidad` |

El backend construye internamente un correo electrónico derivado del identificador (`identificador@unadeca.work.senda`) para autenticarse contra Supabase Auth. Este mecanismo permite que los usuarios inicien sesión con credenciales simplificadas sin necesidad de recordar un correo electrónico.

### 8.4 Características Técnicas

- **Carga diferida (Code Splitting)**: Cada portal se carga con `React.lazy()` y `Suspense`, enviando al navegador únicamente el código necesario para el rol autenticado.
- **Persistencia de sesión**: Al recargar la página, la aplicación intenta restaurar la sesión leyendo el token almacenado en `localStorage`.
- **Animaciones**: Transiciones fluidas con la biblioteca `Motion` al cambiar entre estados (carga, error, portal).
- **Detección de kiosco**: Si existe un kiosco activo para el departamento del usuario que inicia sesión, la pantalla se reemplaza automáticamente por la interfaz de kiosco.

---

## 9. Portal del Estudiante

### 9.1 Descripción General

El portal del estudiante es la interfaz principal para los becarios de UNADECA. Permite registrar horas trabajadas mediante un cronómetro en tiempo real, consultar el historial de registros con sus estados, y visualizar un resumen financiero proyectado del ciclo actual.

### 9.2 Componentes del Portal

El portal se compone de **4 secciones principales**, orquestadas por `StudentPortal.tsx`:

```
┌──────────────────────────────────────────────────────────┐
│  StudentProfile — Perfil del estudiante                  │
│  [Avatar] Nombre Completo — Carnet: 12345                │
│  📊 Horas Totales: 45.5    💰 Tarifa: ₡1,200.00/hora    │
├────────────────────────────────┬─────────────────────────┤
│  StudentHistory                │  StudentTimer           │
│  ┌──────────────────────────┐  │  ┌───────────────────┐  │
│  │ [Mes] [Cuatrimestre]     │  │  │   02:34:15        │  │
│  │ Selector: Mar 2026       │  │  │   ● En progreso   │  │
│  │                          │  │  │                   │  │
│  │ Fecha  Horas  Estado     │  │  │  [▶ Iniciar]      │  │
│  │ 03/10  3.5    ✅ Aprob   │  │  │  [■ Finalizar]    │  │
│  │ 03/08  2.0    🟡 Pend   │  │  └───────────────────┘  │
│  │ 03/05  4.0    ✅ Aprob   │  │                         │
│  │                          │  │  StudentFinancials      │
│  │ [Exportar CSV] [PDF]     │  │  ┌───────────────────┐  │
│  └──────────────────────────┘  │  │ Total Neto ₡52,200│  │
│                                │  │ Diezmo     ₡5,800 │  │
│                                │  │ Próx. Corte: 25/04│  │
│                                │  └───────────────────┘  │
└────────────────────────────────┴─────────────────────────┘
```

### 9.3 Cronómetro en Tiempo Real (`StudentTimer`)

El cronómetro utiliza el hook `useStudentSession`, que implementa las siguientes características:

- **Persistencia**: La sesión activa se almacena en `localStorage`, permitiendo que el cronómetro continúe incluso si el usuario recarga la página o cierra el navegador.
- **Protección anti-sesiones zombie**: Al restaurar una sesión, se verifica que no haya excedido el límite máximo de horas (12). Si lo supera, se descarta automáticamente.
- **Flujo de registro**:
  1. El estudiante presiona **Iniciar** → se guarda `startTime` en localStorage.
  2. El cronómetro muestra el tiempo transcurrido en formato `HH:MM:SS`.
  3. Al presionar **Finalizar**, se solicita una descripción de las tareas realizadas.
  4. Un diálogo de confirmación muestra el resumen antes de enviar.
  5. Se envía el registro al backend con estado `PENDING`.

### 9.4 Historial y Filtros (`StudentHistory`)

- **Vista por Mes**: Muestra registros filtrados por ciclo de facturación (meses individuales, últimos 6 meses disponibles).
- **Vista por Cuatrimestre**: Agrupa registros por cuatrimestre académico (Enero-Abril, Mayo-Agosto, Septiembre-Diciembre).
- **Tabla de registros**: Utiliza el componente compartido `WorkLogTable` con columnas de fecha, horas, descripción y estado (badge coloreado).

### 9.5 Resumen Financiero (`StudentFinancials`)

Presenta cálculos proyectados basados en las horas aprobadas y la tasa por hora vigente:

| Concepto | Fórmula |
|:---------|:--------|
| **Monto Bruto** | `horasAprobadas × tasaPorHora` |
| **Diezmo (10%)** | `montoBruto × 0.10` |
| **Monto Neto** | `montoBruto − diezmo` |
| **Próximo Corte** | Día 25 del mes actual o siguiente |

### 9.6 Exportación de Datos

El estudiante puede exportar su historial en dos formatos:
- **CSV**: Archivo de texto plano con separación por comas.
- **PDF**: Documento formateado con encabezado institucional, datos del estudiante, tabla de registros y resumen financiero, generado con la biblioteca `jsPDF`.

---

## 10. Portal del Jefe de Departamento

### 10.1 Descripción General

El portal del jefe de departamento sirve como centro de gestión para la supervisión y aprobación de las horas trabajadas por los estudiantes becarios asignados a su departamento. Combina indicadores clave, flujo de aprobación y la capacidad de registrar horas directamente en nombre de un estudiante.

### 10.2 Estructura del Portal

```
┌──────────────────────────────────────────────────────────────┐
│  Departamento de [Nombre] — Jefe: [Nombre del Jefe]         │
│  [Selector de Ciclo ▼]    [Activar Kiosco]                  │
├──────────────────────────────────────────────────────────────┤
│  📊 Estudiantes  │  ⏱ Horas Ciclo  │  🟡 Pendientes        │
│       8          │     127.5        │       3                │
│  ✅ Aprobadas    │  💰 Facturación                          │
│       45         │     ₡153,000                             │
├──────────────────────────────────┬───────────────────────────┤
│  Registros Pendientes            │  Registro Rápido (sticky) │
│  ┌────────────────────────────┐  │  ┌─────────────────────┐  │
│  │ [✓ Aprobar Todo]           │  │  │ Estudiante: [▼]     │  │
│  │ Estudiante Fecha Hrs Acción│  │  │ Fecha:     [📅]     │  │
│  │ Juan P.   03/10 3.5 [✓][✗]│  │  │ Horas:     [2.5]    │  │
│  │ María L.  03/09 2.0 [✓][✗]│  │  │ Descripción:        │  │
│  └────────────────────────────┘  │  │ [                 ]  │  │
│                                  │  │ [Registrar y Aprobar]│  │
│  Historial del Departamento      │  └─────────────────────┘  │
│  ┌────────────────────────────┐  │                           │
│  │ Estudiante Fecha Hrs Estado│  │                           │
│  │ ...                        │  │                           │
│  │ [Exportar CSV] [PDF]       │  │                           │
│  └────────────────────────────┘  │                           │
└──────────────────────────────────┴───────────────────────────┘
```

### 10.3 Flujo de Aprobación

El jefe de departamento gestiona el ciclo de vida de los registros de horas mediante las siguientes acciones:

**Aprobación individual**: Un clic en el botón ✓ de cada registro cambia su estado de `PENDING` a `APPROVED`, registrando automáticamente el `approved_by` y `approved_at`.

**Aprobación masiva**: El botón "Aprobar Todo" procesa todos los registros pendientes del ciclo seleccionado en una sola operación, utilizando el endpoint `PATCH /api/v1/work-logs/bulk-status`.

**Rechazo**: El botón ✗ abre un modal que solicita una razón de rechazo obligatoria (con límite de caracteres). El registro cambia a estado `REJECTED` con los campos `rejected_by`, `rejected_at` y `rejection_reason` registrados.

### 10.4 Registro Rápido

El formulario lateral permite al jefe registrar horas directamente en nombre de un estudiante. Este registro se crea con estado `APPROVED` automáticamente, ya que el jefe actúa simultáneamente como registrador y aprobador. Los campos disponibles son:

- **Estudiante**: Selector desplegable con los becarios asignados al departamento.
- **Fecha**: Selector de fecha (por defecto la fecha actual).
- **Horas**: Campo numérico con paso de 0.5, máximo configurable.
- **Descripción**: Área de texto con límite de caracteres para describir las tareas realizadas.

### 10.5 Activación del Kiosco

Desde este portal, el jefe puede activar el **modo kiosco** para su departamento, convirtiendo la computadora en un terminal de fichaje. La activación requiere la autenticación con las credenciales del jefe (número de empleado y contraseña) como medida de seguridad. Una vez activado, toda la interfaz se reemplaza por la pantalla de kiosco a pantalla completa.

---

## 11. Portal del Administrador

### 11.1 Descripción General

El portal del administrador ofrece una vista panorámica de todo el sistema con capacidades completas de gestión. Organizado mediante pestañas de navegación, permite supervisar la actividad global, gestionar cuentas de estudiantes y jefes de departamento, y administrar la estructura departamental de la institución.

### 11.2 Estructura de Pestañas

El portal utiliza un sistema de 4 pestañas con indicadores visuales de la sección activa:

| Pestaña | Contenido |
|:--------|:----------|
| **Dashboard** | Indicadores globales, gráficos estadísticos y tabla de registros con filtros avanzados |
| **Estudiantes** | CRUD completo de cuentas de estudiantes becarios |
| **Jefes Depto.** | CRUD completo de cuentas de jefes de departamento |
| **Departamentos** | Gestión de la estructura departamental (nombre, jefe, centro de costo) |

### 11.3 Dashboard — Vista Global

El dashboard presenta tres niveles de información:

**Tarjetas de Indicadores (KPI)**:
- **Horas del Ciclo**: Total de horas registradas en el período seleccionado.
- **Estudiantes Activos**: Cantidad de becarios con cuenta activa.
- **Total Pago Global**: Monto total calculado para el ciclo actual.

**Gráficos Estadísticos** (Recharts):
- **Gráfico de barras**: Top 5 estudiantes por cantidad de horas registradas.
- **Gráfico circular**: Distribución porcentual de horas por departamento.

**Tabla de Registros Globales** con sistema de filtros avanzado:

| Filtro | Opciones |
|:-------|:---------|
| Búsqueda rápida | Por nombre de estudiante |
| Estado | Todos, Pendientes, Aprobados, Rechazados, Procesados |
| Departamento | Selector con todos los departamentos |
| Fuente | Manual, Kiosco |
| Resolución | Aprobados por jefe, Rechazados |
| Rango de horas | Mínimo — Máximo |
| Ordenamiento | 8 criterios: recientes, horas, estudiante A-Z, pendientes primero, etc. |

Al hacer clic en una fila, se despliega un modal con el detalle completo del registro: datos del estudiante, departamento, jefe responsable, fechas, horas, descripción, estado, y la traza de auditoría (quién aprobó/rechazó y cuándo).

### 11.4 Gestión de Estudiantes

Tabla interactiva con búsqueda en tiempo real y las siguientes columnas:

| Columna | Descripción |
|:--------|:------------|
| Nombre | Nombre completo del estudiante |
| Carnet | Número de identificación institucional |
| Correo | Email institucional (opcional) |
| Departamento | Departamento asignado (badge coloreado) |
| Estado | Activo / Inactivo con indicador visual |
| Acciones | Editar, Activar/Desactivar, Eliminar |

**Formulario de creación/edición**: Modal con campos para nombre, carnet, correo institucional (opcional), departamento (selector) y contraseña temporal (solo en creación).

### 11.5 Gestión de Jefes de Departamento

Estructura idéntica a la de estudiantes, con campos específicos para el rol:
- **Nombre completo** — Nombre del jefe.
- **Nº Empleado** — Identificador institucional del empleado.
- **Correo institucional** — Email de contacto.
- **Departamento** — Departamento que dirigirá.

### 11.6 Gestión de Departamentos

Presentados como tarjetas visuales en una cuadrícula, cada departamento muestra:
- Nombre del departamento.
- Jefe asignado (o "Sin asignar").
- Centro de costos (formato `NN-NN-NN`).
- Estadísticas: cantidad de estudiantes asignados, horas acumuladas en el ciclo.
- Botones de acción: Editar y Eliminar.

### 11.7 Gestión de Tarifa

Un botón en la barra superior permite modificar la tarifa por hora, abriendo un modal con campo numérico. La tarifa se almacena con fecha de vigencia, permitiendo mantener un historial de cambios.

---

## 12. Portal del Super Administrador

### 12.1 Descripción General

El Super Administrador posee el nivel de acceso más alto del sistema. Su portal se centra en la gestión de cuentas privilegiadas (administradores y contabilidad), el soporte a estudiantes con problemas de acceso, y la capacidad de operar el kiosco de forma remota desde cualquier departamento.

### 12.2 Estructura del Portal

```
┌──────────────────────────────────────────────────────────┐
│  🛡 Panel de Super Administrador                         │
├──────────────────────────────────┬───────────────────────┤
│  Cuentas Administrativas        │  Crear Cuenta          │
│  ┌────────────────────────────┐  │  ┌─────────────────┐  │
│  │ 🔍 Buscar...               │  │  │ [Admin][Contab] │  │
│  │ Nombre    Rol      Acciones│  │  │ Nombre: [    ]  │  │
│  │ Admin1    ADMIN    [Reset] │  │  │ Contraseña:[  ] │  │
│  │ Conta1    ACCOUNT  [Reset] │  │  │ [Crear Cuenta]  │  │
│  │ Jefe1     DEPT_HD  [Reset] │  │  └─────────────────┘  │
│  └────────────────────────────┘  │                       │
│                                  │  Kiosco Remoto         │
│  Ayuda a Estudiantes             │  ┌─────────────────┐  │
│  ┌────────────────────────────┐  │  │ Depto: [▼]      │  │
│  │ 🔍 Buscar por nombre/carnet│  │  │ Credenciales... │  │
│  │ Nombre    Carnet   Acciones│  │  │ [Activar]       │  │
│  │ Juan P.   12345   [Reset]  │  │  └─────────────────┘  │
│  └────────────────────────────┘  │                       │
└──────────────────────────────────┴───────────────────────┘
```

### 12.3 Gestión de Cuentas Privilegiadas

La tabla de cuentas administrativas muestra todos los usuarios con roles `ADMIN`, `DEPT_HEAD` y `ACCOUNTING`. Para cada cuenta, el Super Administrador puede:

- Visualizar nombre, rol (con badge de color diferenciado) y número de empleado.
- **Resetear contraseña**: Solicita una nueva contraseña temporal (mínimo 8 caracteres) mediante un modal seguro.

### 12.4 Creación de Cuentas

El formulario lateral permite crear cuentas de tipo:
- **Administrador** (`ADMIN`): Acceso completo a la gestión de usuarios y departamentos.
- **Contabilidad** (`ACCOUNTING`): Acceso al módulo de nómina y configuración contable.

Campos requeridos: nombre completo y contraseña temporal. El sistema genera automáticamente el identificador de login basado en el tipo de cuenta.

### 12.5 Soporte a Estudiantes

Una lista scrollable con búsqueda permite localizar estudiantes por nombre o carnet para restablecer sus contraseñas. Esta funcionalidad es esencial para resolver problemas de acceso sin necesidad de que el estudiante acuda físicamente a la oficina administrativa.

### 12.6 Activación Remota de Kiosco

El Super Administrador puede activar el modo kiosco de **cualquier departamento** de forma remota, sin necesidad de estar físicamente en la computadora del departamento. El proceso requiere:
1. Seleccionar el departamento destino.
2. Ingresar las credenciales de Super Administrador como verificación de identidad.
3. Confirmar la activación.

---

## 13. Portal de Contabilidad

### 13.1 Descripción General

El portal de contabilidad es el módulo más complejo del sistema, diseñado para el procesamiento de la nómina de becarios. Ofrece reportes detallados por departamento, cálculos financieros automatizados, gráficos analíticos, y la capacidad de generar archivos de exportación compatibles con el sistema contable institucional.

### 13.2 Estructura del Portal

```
┌──────────────────────────────────────────────────────────────┐
│  🔍 Buscar...  │ Depto: [▼] │ [Mes][Cuatri] │ Ciclo [▼]    │
│  [⚙ Config] [Procesar Pagos] [▼ Exportar: CSV/PDF/TXT]      │
├──────────────────────────────────────────────────────────────┤
│  ⏱ Horas  │ 💰 Bruto    │ 📊 Diezmo   │ 💵 Neto           │
│   245.5   │  ₡294,600   │  ₡29,460    │  ₡265,140         │
│  📋 Por Cobrar │ Total a Pagar                               │
│   ₡15,000      │  ₡250,140                                  │
├──────────────────────────────────────────────────────────────┤
│  [Gráficos]                                                  │
│  ┌─────────────────────┐  ┌─────────────────────┐           │
│  │ Facturado por Depto  │  │ Top 5 Estudiantes   │           │
│  │ ████████ Biblioteca  │  │ ▓▓▓▓▓ Juan P.       │           │
│  │ ██████   IT          │  │ ▓▓▓▓  María L.      │           │
│  │ ████     Manten.     │  │ ▓▓▓   Pedro R.      │           │
│  └─────────────────────┘  └─────────────────────┘           │
├──────────────────────────────────────────────────────────────┤
│  [Tab: Nómina por Depto] [Tab: Resumen]                      │
│                                                              │
│  ▼ Departamento de Biblioteca (4 estudiantes)                │
│  ┌──────────────────────────────────────────────────┐        │
│  │ ☑  Nombre    Carnet  Hrs   Bruto   Diezmo  Neto │ Cobrar │
│  │ ✓  Juan P.   12345   45.5  54,600  5,460  49,140  0     │
│  │ ✓  María L.  12346   30.0  36,000  3,600  32,400  5,000 │
│  │────────────────────────────────────────────────────│      │
│  │ TOTAL                75.5  90,600  9,060  81,540  5,000 │
│  └──────────────────────────────────────────────────┘        │
└──────────────────────────────────────────────────────────────┘
```

### 13.3 Indicadores Financieros (KPI)

El portal presenta 6 tarjetas de indicadores clave calculados en tiempo real:

| Indicador | Fórmula |
|:----------|:--------|
| **Total Horas** | Suma de horas aprobadas en el período |
| **Monto Bruto** | `totalHoras × tasaVigente` |
| **Diezmo (10%)** | `montoBruto × 0.10` |
| **Monto Neto** | `montoBruto − diezmo` |
| **Cuentas por Cobrar** | Suma de montos pendientes de cobro a estudiantes |
| **Total a Pagar** | `montoNeto − cuentasPorCobrar` |

### 13.4 Gráficos Analíticos

El módulo incluye 4 visualizaciones generadas con la biblioteca **Recharts**:

1. **Facturado por Departamento** — Gráfico de barras horizontal mostrando el monto bruto de cada departamento.
2. **Top 5 Estudiantes** — Gráfico de barras vertical con los estudiantes de mayor facturación.
3. **Actividad Semanal** — Lista que muestra la distribución de horas y montos por semana.
4. **Resumen por Cuatrimestre** — Tres tarjetas comparativas con totales de cada cuatrimestre del año.

### 13.5 Nómina por Departamento

La pestaña principal presenta **tarjetas colapsables** para cada departamento que tiene actividad en el período. Cada tarjeta incluye:

- **Encabezado**: Nombre del departamento, cantidad de estudiantes, totales del período.
- **Tabla de estudiantes**: Una fila por estudiante con:
  - Checkbox "Registrado" (estado persistido en `localStorage`).
  - Nombre (clickeable → abre modal de detalle).
  - Carnet, horas, monto bruto, diezmo, neto.
  - **Campo "Por Cobrar"** editable: permite registrar cuentas por cobrar directamente en la tabla.
  - **"Por Pagar"**: Monto neto menos las cuentas por cobrar.
- **Pie de tabla**: Totales del departamento.

### 13.6 Procesamiento de Pagos

El botón "Procesar y Cerrar" ejecuta una operación masiva que:
1. Cambia el estado de todos los registros `APPROVED` del período a `PROCESSED`.
2. Utiliza el endpoint `PATCH /api/v1/work-logs/bulk-status`.
3. Requiere confirmación mediante diálogo de seguridad.
4. Una vez procesados, los registros no pueden ser revertidos.

### 13.7 Configuración Contable

El modal de configuración permite definir las cuentas contables utilizadas en los reportes y exportaciones:

| Cuenta | Descripción |
|:-------|:------------|
| **Becas** | Código y nombre de la cuenta de becas institucional |
| **Diezmo** | Código y nombre de la cuenta de diezmo |
| **Cuentas por Pagar** | Cuenta de pasivo para obligaciones pendientes |
| **Cuentas por Cobrar** | Cuenta de activo para montos pendientes de estudiantes |

Adicionalmente, muestra una cuadrícula de **centros de costo** por departamento (formato `NN-NN-NN`), con alerta visual cuando un departamento carece de centro de costo configurado.

### 13.8 Exportaciones

El portal de contabilidad ofrece 4 formatos de exportación:

| Formato | Contenido |
|:--------|:----------|
| **CSV** | Datos tabulares de la nómina para análisis en hojas de cálculo |
| **PDF Nómina** | Documento con detalle completo por departamento y estudiante |
| **PDF Resumen** | Tabla resumen por departamento con totales globales |
| **TXT Contable** | Archivo de ancho fijo para importación directa al sistema contable institucional, con partidas dobles (débito/crédito) validadas |

El archivo TXT contable es la pieza clave de integración con el sistema ERP de la institución, generando asientos contables con partida doble que incluyen las cuentas de becas, diezmo, cuentas por pagar y cuentas por cobrar.

---

## 14. Módulo de Kiosco

### 14.1 Descripción General

El módulo de kiosco transforma una computadora del departamento en un terminal de fichaje a pantalla completa. Los estudiantes registran su entrada y salida utilizando sus credenciales (carnet y contraseña), y el sistema calcula automáticamente las horas trabajadas, generando un registro con estado `PENDING` listo para aprobación.

### 14.2 Diseño de Interfaz

El kiosco utiliza un tema oscuro exclusivo diseñado para terminales compartidos, con tipografía grande y controles simplificados:

```
┌──────────────────────────────────────────────────────────────┐
│  ● Turno activo    3 en servicio    [⚙ Turnos] [⏻ Desactivar]│
├────────────────────────────────┬─────────────────────────────┤
│  Registro de Entrada/Salida    │  Trabajando ahora            │
│                                │                              │
│  ┌──────────────────────────┐  │  ┌────────────────────────┐  │
│  │  Carnet:                 │  │  │ 👤 Juan Pérez          │  │
│  │  ┌────────────────────┐  │  │  │    Carnet: 12345       │  │
│  │  │                    │  │  │  │    ⏱ 02:34:15          │  │
│  │  └────────────────────┘  │  │  │              [Cancelar]│  │
│  │  Contraseña:             │  │  ├────────────────────────┤  │
│  │  ┌────────────────────┐  │  │  │ 👤 María López         │  │
│  │  │  ••••••••          │  │  │  │    Carnet: 12346       │  │
│  │  └────────────────────┘  │  │  │    ⏱ 01:15:42          │  │
│  │                          │  │  │              [Cancelar]│  │
│  │  [     Registrar     ]   │  │  └────────────────────────┘  │
│  │                          │  │                              │
│  │  ⓘ Su contraseña no se  │  │         [Pantalla vacía      │
│  │    almacena en este      │  │          cuando no hay       │
│  │    dispositivo           │  │          sesiones activas]   │
│  └──────────────────────────┘  │                              │
└────────────────────────────────┴─────────────────────────────┘
```

### 14.3 Flujo de Fichaje

El kiosco implementa un sistema inteligente de detección de estado:

**Entrada (Clock-In)**:
1. El estudiante ingresa carnet y contraseña.
2. El sistema verifica las credenciales contra Supabase Auth.
3. Si no tiene sesión activa, se crea una nueva sesión con `started_at`.
4. El nombre del estudiante aparece en el tablero "Trabajando ahora" con un cronómetro en tiempo real.

**Salida (Clock-Out)**:
1. El mismo estudiante ingresa su carnet y contraseña de nuevo.
2. El sistema detecta que ya tiene una sesión activa.
3. Calcula automáticamente las horas trabajadas (`end_time - start_time`).
4. Crea un registro `work_log` con estado `PENDING` y fuente `KIOSK`.
5. La sesión se cierra y desaparece del tablero.

**Cancelación de Sesión**:
1. El jefe de departamento puede cancelar una sesión activa desde el tablero.
2. Requiere sus credenciales (número de empleado + contraseña) y una razón.
3. Se crea un registro con estado `REJECTED` y el motivo de cancelación.

### 14.4 Gestión de Turnos

El kiosco permite configurar **turnos programados** que definen las ventanas de tiempo en que los estudiantes pueden fichar:

- Cada turno define un rango horario (hora de inicio → hora de fin).
- Se pueden agregar múltiples turnos por día (ej. mañana y tarde).
- La modificación de turnos requiere credenciales del jefe de departamento.
- El indicador visual "Turno activo" se muestra cuando el horario actual está dentro de un turno programado.

### 14.5 Desactivación del Kiosco

El kiosco puede ser desactivado por el jefe de departamento o el Super Administrador. Al desactivar:
1. Se requieren credenciales de autorización.
2. Todas las sesiones activas se finalizan automáticamente (flush).
3. La pantalla retorna al portal normal del usuario.
4. Los registros de las sesiones finalizadas se crean con estado `PENDING`.

---

## 15. Actualización en Tiempo Real

### 15.1 Arquitectura de Tiempo Real

SENDA implementa actualizaciones en tiempo real mediante **Supabase Realtime**, un servicio basado en WebSockets que transmite cambios de la base de datos a los clientes conectados. Esto permite que múltiples usuarios vean reflejados los cambios de otros sin necesidad de recargar la página.

### 15.2 Tablas con Replicación Habilitada

La migración `004_enable_realtime_for_core_tables` habilita la replicación en tiempo real para las siguientes tablas:

| Tabla | Justificación |
|:------|:-------------|
| `profiles` | Los cambios en usuarios (activación/desactivación, cambio de rol) se reflejan inmediatamente |
| `departments` | La creación o modificación de departamentos se propaga a todos los portales |
| `work_logs` | Los registros de horas se actualizan en vivo en el portal del jefe, admin y contabilidad |
| `hourly_rates` | Los cambios de tarifa se reflejan instantáneamente en los cálculos financieros |

### 15.3 Patrón de Suscripción

Todos los hooks que requieren datos en tiempo real implementan un patrón estandarizado documentado en las instrucciones del proyecto:

```
useEffect (suscripción) → subscribeToTableChanges()
    │
    └──▶ onChange → scheduleBackgroundRefresh()
              │
              └──▶ setTimeout(250ms) → fetchLatestData(paramsRef.current)
                        │
                        └──▶ setData(result)  // sin isLoading = true
```

**Principios clave del patrón:**

1. **Suscripción única**: El `useEffect` de suscripción se monta una sola vez (dependencias vacías `[]`).
2. **Debounce de 250ms**: Un timer evita que ráfagas de eventos sobrecarguen el servidor con peticiones.
3. **Parámetros por referencia**: Un `useRef` almacena los filtros actuales, permitiendo que el callback siempre use los valores más recientes sin re-montar la suscripción.
4. **Actualización silenciosa**: El refresh de realtime **no** activa indicadores de carga — la UI se actualiza en segundo plano sin bloquear la interacción del usuario.
5. **Degradación elegante**: Si las variables de entorno de Supabase no están configuradas, la suscripción se desactiva automáticamente sin errores.

### 15.4 Hooks con Suscripción Activa

| Hook | Tabla escuchada | Datos actualizados |
|:-----|:----------------|:-------------------|
| `useWorkLogs` | `work_logs` | Registros de horas en todos los portales |
| `useAccountingReport` | `work_logs` | Cálculos financieros de contabilidad |

---

## 16. Lógica de Negocio

### 16.1 Ciclos de Facturación

El sistema organiza los registros de horas en **ciclos de facturación mensuales** con corte el día 25 de cada mes. La lógica de ciclos está implementada en `frontend/lib/business.ts` y `backend/src/features/workLogs/workLogs.schemas.mjs`.

**Definición de ciclo**:
- Un ciclo va del **día 26 del mes anterior** al **día 25 del mes actual**.
- Ejemplo: El ciclo "Marzo 2026" abarca del 26 de febrero al 25 de marzo de 2026.

```
       Feb 26              Mar 25
         │                    │
         ▼                    ▼
    ─────┤ Ciclo Marzo 2026  ├─────
         │◀───────────────────▶│
```

### 16.2 Cuatrimestres Académicos

Para reportes de largo plazo, el sistema agrupa los ciclos en cuatrimestres:

| Cuatrimestre | Meses incluidos | Ciclos de facturación |
|:-------------|:----------------|:---------------------|
| **I** (Enero-Abril) | Enero, Febrero, Marzo, Abril | Ene 26 – Abr 25 |
| **II** (Mayo-Agosto) | Mayo, Junio, Julio, Agosto | May 26 – Ago 25 |
| **III** (Septiembre-Diciembre) | Septiembre, Octubre, Noviembre, Diciembre | Sep 26 – Dic 25 |

### 16.3 Cálculo de Nómina

El proceso de cálculo de nómina sigue la siguiente fórmula para cada estudiante en un período dado:

```
horasAprobadas  = Σ work_logs.hours  WHERE status = 'APPROVED'
montoBruto      = horasAprobadas × tasaVigente
diezmo          = montoBruto × 0.10
montoNeto       = montoBruto − diezmo
cuentasPorCobrar = student_receivables.amount  (si existe para el período)
totalAPagar     = montoNeto − cuentasPorCobrar
```

### 16.4 Ciclo de Vida de un Registro

Cada registro de horas (`work_log`) sigue un ciclo de vida estricto representado por su campo `status`:

```
                    ┌─────────────────┐
   Estudiante       │                 │     Jefe de
   registra    ────▶│    PENDING      │────▶ Departamento
   horas            │                 │     revisa
                    └────────┬────────┘
                             │
                    ┌────────┴────────┐
                    │                 │
              ┌─────▼─────┐    ┌─────▼─────┐
              │  APPROVED │    │  REJECTED  │
              │           │    │  + razón   │
              └─────┬─────┘    └────────────┘
                    │
                    │  Contabilidad
                    │  procesa pago
                    ▼
              ┌───────────┐
              │ PROCESSED │
              │ (final)   │
              └───────────┘
```

**Transiciones permitidas**:
- `PENDING` → `APPROVED`: Por el jefe del departamento (individual o masivo).
- `PENDING` → `REJECTED`: Por el jefe del departamento (requiere razón).
- `APPROVED` → `PROCESSED`: Por contabilidad (operación masiva de cierre).

### 16.5 Fuentes de Registro

Los registros de horas pueden originarse de dos fuentes, identificadas por el campo `entry_source`:

| Fuente | Descripción | Campos adicionales |
|:-------|:------------|:-------------------|
| `MANUAL` | Registro manual desde el portal del estudiante o el formulario del jefe | Solo `date` y `hours` |
| `KIOSK` | Registro automático por fichaje en el kiosco | `start_time`, `end_time` (horas calculadas automáticamente) |

### 16.6 Validaciones de Negocio

El sistema aplica las siguientes reglas de validación en la capa de servicio del backend:

| Regla | Validación |
|:------|:-----------|
| **Horas por registro** | Mínimo 0.1, máximo 12 horas |
| **Descripción** | Obligatoria, máximo 200 caracteres |
| **Razón de rechazo** | Obligatoria al rechazar, máximo 500 caracteres |
| **Departamento** | El estudiante debe pertenecer al departamento del registro |
| **Jefe aprobador** | Solo puede aprobar/rechazar registros de su propio departamento |
| **Tasa por hora** | Debe ser mayor a 0 |
| **Centro de costo** | Formato `NN-NN-NN` (2 dígitos guion 2 dígitos guion 2 dígitos) |
| **Sesión de kiosco** | Un estudiante no puede tener dos sesiones activas simultáneas |

---

## 17. Conclusiones y Trabajo Futuro

### 17.1 Objetivos Alcanzados

El sistema SENDA cumple con los objetivos establecidos para la primera versión:

1. **Digitalización completa** del proceso de registro y aprobación de horas becarias, eliminando los formularios en papel y las hojas de cálculo manuales que generaban errores y pérdida de datos.

2. **Control de acceso basado en roles**: Cinco roles claramente definidos con permisos granulares, respaldados por Row-Level Security a nivel de base de datos y verificación en la capa de servicio del backend.

3. **Trazabilidad total**: Cada registro de horas incluye campos de auditoría que identifican quién lo creó, quién lo aprobó o rechazó, cuándo ocurrió cada acción, y el motivo en caso de rechazo.

4. **Módulo de kiosco innovador**: Terminal de fichaje a pantalla completa que permite a los estudiantes registrar entrada y salida con cronómetro, eliminando la necesidad de calcular horas manualmente.

5. **Integración contable**: Generación de archivos de exportación en formato de ancho fijo para importación directa al sistema ERP institucional, incluyendo asientos de partida doble.

6. **Actualización en tiempo real**: Los cambios en registros se reflejan instantáneamente en todos los portales conectados mediante WebSockets.

### 17.2 Métricas del Proyecto

| Métrica | Valor |
|:--------|:------|
| Líneas de código (frontend) | ~12,000 |
| Líneas de código (backend) | ~4,000 |
| Componentes React | ~40 |
| Hooks personalizados | 15 |
| Endpoints API | 31 |
| Tablas de base de datos | 8 |
| Migraciones SQL | 7 |
| Roles de usuario | 5 |
| Formatos de exportación | 4 (CSV, PDF, PDF Resumen, TXT) |

### 17.3 Trabajo Futuro

Para versiones futuras del sistema se plantean las siguientes mejoras:

- **Notificaciones push**: Alertas en tiempo real cuando se aprueba, rechaza o procesa un registro.
- **Aplicación móvil**: Versión nativa para dispositivos móviles con lectura de código QR para fichaje.
- **Reportes avanzados**: Dashboards con gráficos de tendencia, comparativas entre períodos y exportación a formatos adicionales.
- **Integración con calendario académico**: Sincronización automática con el calendario de la institución para ajustar los ciclos de facturación.
- **Firma digital**: Implementación de firma electrónica para la aprobación de registros con valor legal.
- **Multi-idioma**: Soporte para interfaz en inglés además de español.

### 17.4 Lecciones Aprendidas

El desarrollo del proyecto dejó aprendizajes significativos en:

- La importancia de definir **convenciones de arquitectura** desde el inicio (Feature-Based Architecture, capas de responsabilidad) para mantener la escalabilidad del código.
- El valor de **Row-Level Security** como segunda línea de defensa, más allá de la validación en el backend.
- La necesidad de **patrones estandarizados** para funcionalidades recurrentes como las suscripciones en tiempo real.
- La complejidad inherente al módulo contable, que requirió colaboración estrecha con el departamento financiero para definir correctamente las reglas de partida doble y los formatos de exportación.

---

*Documento generado como parte de la defensa académica del proyecto SENDA — Sistema de Control de Horas Becarias para UNADECA.*

*Abril 2026*